## Pipeline for the image classification

In [1]:
###### NB. This notebook can be run in one go or can be exported to a .py file and will also run ######
##### NB. This will copy the images and save them to a new dir where it then randomly samples and arranges them into train, val, test folders #####
#### NB. Produces model performance metrics in a .csv, Gradcam images and saliency map images ####
### NB. the VGG16 additonal layers were tuned using 'keras-tuner' ###

In [2]:
# Standard library imports
import os
import random
import math
import shutil
from datetime import datetime

# Data science imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from sklearn.metrics import confusion_matrix, classification_report

# Data augmentation
import Augmentor

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend, optimizers, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import (
    ModelCheckpoint, 
    LearningRateScheduler, 
    TensorBoard, 
    EarlyStopping
)
from tensorflow.keras.layers import (
    Conv2D, 
    MaxPooling2D, 
    ZeroPadding2D, 
    Activation, 
    Flatten, 
    Dense, 
    Dropout, 
    GlobalAveragePooling2D, 
    BatchNormalization
)
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

# Keras Tuner
import keras_tuner as kt

# Visualization tools
from tf_keras_vis.gradcam import Gradcam
from tf_keras_vis.saliency import Saliency
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear

# Initialize timestamp
date = datetime.now().strftime('%Y_%m_%d-%I:%M_%S_%p')


In [3]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # or set it to '3' to suppress all messages, including INFO and WARNING

print("Number of GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


Number of GPUs Available:  2


In [4]:

# Seed for reproducibility
SEED = 666

# Function to initialize seeds for all libraries which might have stochastic behavior
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)

# Function to ensure partial determinism
def set_partial_determinism(seed=SEED):
    set_seeds(seed=seed)
    
    # Comment out the line below if you are facing issues with deterministic operations
    # os.environ['TF_DETERMINISTIC_OPS'] = '1'
    
    # Use the following lines if you want to use CPU for deterministic operations
    # tf.config.set_visible_devices([], 'GPU')

# Call the above function with seed value
set_partial_determinism(seed=SEED)

# Ensure XLA is disabled
tf.config.optimizer.set_jit(False)


In [5]:


# # Define the path to the root directory where the subdirectories are located
# root_dir = './dataset_A_seg_split/'     


# # Define the path to the new directory to create
# new_dir = './dataset_A_seg_split/'





In [6]:
##### NB. I set this up to copy only subdirs with at least 150 images to new dir. Good for pruning small classes #####


# # Create the new directory if it does not exist
# if not os.path.exists(new_dir):
#     os.makedirs(new_dir)

# # Iterate through the subdirectories in the root directory
# for subdir in os.listdir(root_dir):
#     # Construct the full path to the subdirectory
#     subdir_path = os.path.join(root_dir, subdir)
    
#     # Check if the subdirectory contains at least 200 images
#     if len(os.listdir(subdir_path)) >= 150: ## Adjust the number as needed. Set to 1 if you don't want this filter
#         # Copy the subdirectory and its contents to the new directory
#         shutil.copytree(subdir_path, os.path.join(new_dir, subdir))


In [7]:
# ##### NB. The following code prunes images from specific species-location combinations to ensure no more than 100 images per combination #####

# pruned_dir = "../save_offs/FV_dorsal/pt2/pruned_directory/"
# if not os.path.exists(pruned_dir):
#     os.makedirs(pruned_dir)

# # Species and their location codes for pruning
# prune_specs = {
#     'fv': ['pa'],
#     'lc': ['ba', 'pa', 'sj'],
#     'ls': ['pdc', 'pm']
# }

# # Iterate through the subdirectories in the new directory
# for subdir in os.listdir(new_dir):
#     subdir_path = os.path.join(new_dir, subdir)
    
#     # Ensure the path is a directory
#     if os.path.isdir(subdir_path):
#         # Iterate through the files in each subdirectory
#         species_images = {}
        
#         for filename in os.listdir(subdir_path):
#             # Check if the file is an image
#             if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
#                 # Extract species code and location code from filename
#                 parts = filename.split('_')
#                 if len(parts) >= 3:
#                     species_code = parts[0]
#                     location_code = parts[1]

#                     # Check if the species needs pruning at this location
#                     if species_code in prune_specs and location_code in prune_specs[species_code]:
#                         # Add the image path to the dictionary for this species-location combination
#                         key = f"{species_code}_{location_code}"
#                         if key not in species_images:
#                             species_images[key] = []
#                         species_images[key].append(os.path.join(subdir_path, filename))
        
#         # Prune images to move any excess images beyond 100 per species-location
#         for key, images in species_images.items():
#             if len(images) > 100:
#                 # Randomly select images to move to pruned directory
#                 images_to_move = random.sample(images, len(images) - 100)

#                 # Create a directory in pruned_dir to save the pruned images
#                 species_code, location_code = key.split('_')
#                 save_dir = os.path.join(pruned_dir, species_code, location_code)
#                 if not os.path.exists(save_dir):
#                     os.makedirs(save_dir)

#                 # Move the excess images to the pruned directory
#                 for image in images_to_move:
#                     shutil.move(image, save_dir)

In [8]:

# # Define source directory and destination directory paths
# src_dir = "../save_offs/FV_dorsal/pt1"  # Update to the correct source directory path
# dst_dir = "../save_offs/FV_dorsal/pt2"

# # Create train, validation, and test directories if they do not exist
# train_dir = os.path.join(dst_dir, "train")
# val_dir = os.path.join(dst_dir, "validation")
# test_dir = os.path.join(dst_dir, "test")
# for d in [train_dir, val_dir, test_dir]:
#     if not os.path.exists(d):
#         os.makedirs(d)

# # Loop through subdirectories in source directory
# for subdir in os.listdir(src_dir):
#     subdir_path = os.path.join(src_dir, subdir)
#     if os.path.isdir(subdir_path):
#         # Create subdirectories in train, validation, and test directories with identical names
#         train_subdir = os.path.join(train_dir, subdir)
#         val_subdir = os.path.join(val_dir, subdir)
#         test_subdir = os.path.join(test_dir, subdir)
#         for d in [train_subdir, val_subdir, test_subdir]:
#             if not os.path.exists(d):
#                 os.makedirs(d)

#         # Get list of image files in the subdirectory
#         image_files = [f for f in os.listdir(subdir_path) if f.endswith(".jpg")]

#         # Specify number of training and validation samples
#         num_train_samples = min(100, len(image_files))
#         num_val_samples = min(25, len(image_files) - num_train_samples)

#         # Randomly sample images for train, validation, and test sets
#         num_total_samples = len(image_files)
#         num_test_samples = num_total_samples - num_train_samples - num_val_samples
#         train_samples = random.sample(image_files, num_train_samples)
#         remaining_files = list(set(image_files) - set(train_samples))
#         val_samples = random.sample(remaining_files, num_val_samples)
#         remaining_files = list(set(remaining_files) - set(val_samples))
#         test_samples = random.sample(remaining_files, num_test_samples)

#         # Move images to train, validation, and test subdirectories
#         for train_sample in train_samples:
#             src_path = os.path.join(subdir_path, train_sample)
#             dst_path = os.path.join(train_subdir, train_sample)
#             shutil.move(src_path, dst_path)

#         for val_sample in val_samples:
#             src_path = os.path.join(subdir_path, val_sample)
#             dst_path = os.path.join(val_subdir, val_sample)
#             shutil.move(src_path, dst_path)

#         for test_sample in test_samples:
#             src_path = os.path.join(subdir_path, test_sample)
#             dst_path = os.path.join(test_subdir, test_sample)
#             shutil.move(src_path, dst_path)



In [9]:


# # Create a new directory called "test_even" within the destination directory
# test_dir = os.path.join(dst_dir, "test")
# test_even_dir = os.path.join(dst_dir, "test_even")
# if not os.path.exists(test_even_dir):
#     os.makedirs(test_even_dir)




In [10]:
# # Define paths
# pruned_dir = "../save_offs/FV_dorsal/pt2/pruned_directory/"
# test_dir = "../save_offs/FV_dorsal/pt2/test/"

# # Ensure the pruned directory exists
# if not os.path.exists(pruned_dir):
#     raise FileNotFoundError(f"Pruned directory does not exist: {pruned_dir}")

# # Get a list of test subdirectories that start with 'south_'
# subdirs = [d for d in os.listdir(test_dir) if d.startswith('south_') and os.path.isdir(os.path.join(test_dir, d))]

# if not subdirs:
#     print("No target subdirectories found in the test directory. Ensure that folders starting with 'south_' are present.")

# # Organize pruned images back into the appropriate test set folders
# for root, dirs, files in os.walk(pruned_dir):
#     for filename in files:
#         if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
#             # Extract the classification from the filename
#             target_subdir = None
#             if 'dorsal' in filename.lower():
#                 target_subdir = next((d for d in subdirs if 'dorsal' in d.lower()), None)
#             elif 'ventral' in filename.lower():
#                 target_subdir = next((d for d in subdirs if 'ventral' in d.lower()), None)

#             # If a matching subdirectory is found, move the image there
#             if target_subdir:
#                 src_path = os.path.join(root, filename)
#                 dst_path = os.path.join(test_dir, target_subdir, filename)

#                 # Check if the destination file already exists
#                 if os.path.exists(dst_path):
#                     print(f"File already exists at destination, renaming: {dst_path}")
#                     base, ext = os.path.splitext(filename)
#                     new_filename = f"{base}_pruned{ext}"
#                     dst_path = os.path.join(test_dir, target_subdir, new_filename)

#                 try:
#                     # Move the file to the destination directory
#                     print(f"Attempting to move {src_path} to {dst_path}")
#                     shutil.move(src_path, dst_path)
#                     print(f"Moved {src_path} to {dst_path}")
#                 except PermissionError as pe:
#                     print(f"Permission error while moving {src_path} to {dst_path}: {pe}")
#                 except FileNotFoundError as fnfe:
#                     print(f"File not found error while moving {src_path} to {dst_path}: {fnfe}")
#                 except Exception as e:
#                     print(f"Failed to move {src_path} to {dst_path}: {e}")
#             else:
#                 print(f"No matching test subdirectory found for {filename}")

# # Verify that no files remain in the pruned directory
# def verify_pruned_directory_empty(pruned_dir):
#     for root, dirs, files in os.walk(pruned_dir):
#         if files:
#             print(f"Files still remaining in pruned directory: {root}")
#             for file in files:
#                 print(f" - {file}")
#         else:
#             print(f"No files remaining in pruned directory: {root}")

# verify_pruned_directory_empty(pruned_dir)


In [11]:
# # Loop through subdirectories in test directory
# for subdir in os.listdir(test_dir):
#     subdir_path = os.path.join(test_dir, subdir)
#     if os.path.isdir(subdir_path):
#         # Create identical subdirectories within test_even directory
#         test_even_subdir = os.path.join(test_even_dir, subdir)
#         if not os.path.exists(test_even_subdir):
#             os.makedirs(test_even_subdir)

#         # Randomly sample 20 images from the subdirectory
#         image_files = [f for f in os.listdir(subdir_path) if f.endswith(".jpg")]
#         num_images = len(image_files)
#         if num_images < 20:
#             num_samples = num_images
#         else:
#             num_samples = 20
#         sample_files = random.sample(image_files, num_samples)

#         # Copy sampled images to the test_even subdirectory
#         for sample_file in sample_files:
#             src_path = os.path.join(subdir_path, sample_file)
#             dst_path = os.path.join(test_even_subdir, sample_file)
#             shutil.copy(src_path, dst_path)

In [12]:
# dir = '../save_offs/FV_dorsal/pt2/train/'

# # Get a list of all subdirectories within the main directory
# subdirs = [d for d in os.listdir(dir) if os.path.isdir(os.path.join(dir, d))]

# # Loop over each subdirectory and apply augmentations
# for subdir in subdirs:
#     subdir_path = os.path.join(dir, subdir)
#     p = Augmentor.Pipeline(subdir_path, output_directory='')
#     p.flip_left_right(probability=0.5)
#     p.flip_top_bottom(probability=0.5)
#     #p.rotate(probability=0.9,max_left_rotation=18,max_right_rotation=18)
#     p.rotate90(probability=0.5)
#     p.rotate180(probability=0.5)
#     p.rotate270(probability=0.5)
#     #p.scale(probability=0.5, scale_factor=1.5)
#     p.random_brightness(probability=0.5,min_factor=0.5,max_factor=1.5)
#     p.random_contrast(probability=0.5,min_factor=0.5,max_factor=1.5)

#     # Get the number of files in the directory
#     num_files = len(os.listdir(subdir_path))

#     # Sample additional images if necessary
#     num_samples = 2400 - num_files
#     if num_samples > 0:
#         p.sample(num_samples)


In [13]:
# dir = '../save_offs/FV_dorsal/pt2/validation/'

# # Get a list of all subdirectories within the main directory
# subdirs = [d for d in os.listdir(dir) if os.path.isdir(os.path.join(dir, d))]

# # Loop over each subdirectory and apply augmentations
# for subdir in subdirs:
#     subdir_path = os.path.join(dir, subdir)
#     p = Augmentor.Pipeline(subdir_path, output_directory='')
#     p.flip_left_right(probability=0.5)
#     p.flip_top_bottom(probability=0.5)
#     #p.rotate(probability=0.9,max_left_rotation=18,max_right_rotation=18)
#     p.rotate90(probability=0.5)
#     p.rotate180(probability=0.5)
#     p.rotate270(probability=0.5)
#     #p.scale(probability=0.5, scale_factor=1.5)
#     p.random_brightness(probability=0.5,min_factor=0.5,max_factor=1.5)
#     p.random_contrast(probability=0.5,min_factor=0.5,max_factor=1.5)

#     # Get the number of files in the directory
#     num_files = len(os.listdir(subdir_path))

#     # Sample additional images if necessary
#     num_samples = 600 - num_files
#     if num_samples > 0:
#         p.sample(num_samples)

In [14]:
# Define the paths to your data directories
train_dir = './dataset_C_seg_split/train/'
validation_dir = './dataset_C_seg_split/validation/'
test_dir = './dataset_C_seg_split/test_even/' 
test_full_dir = './dataset_C_seg_split/test/'


img_width, img_height = 224, 224
BATCHSIZE = 16

# Define a function to preprocess the images
def preprocess_image(image):
    image = tf.image.resize(image, (img_width, img_height))
    image = image / 255.0
    return image


In [15]:
# Load the train data
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_image)

val_datagen = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_image)

In [16]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=BATCHSIZE,
    class_mode='categorical',
    shuffle=True)

# Load the validation data

validation_generator = val_datagen.flow_from_directory(
    validation_dir,
    target_size=(img_width, img_height),
    batch_size=BATCHSIZE,
    class_mode='categorical',
    shuffle=False
)

# Load the test data
test_datagen = keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_image)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_width, img_height),
    batch_size=BATCHSIZE,
    class_mode='categorical',
    shuffle=False
)



test_full_datagen = keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_image)
test_full_generator = test_full_datagen.flow_from_directory(
    test_full_dir,
    target_size=(img_width, img_height),
    batch_size=BATCHSIZE,
    class_mode='categorical',
    shuffle=False
)


# Store the class names in a list
class_names = list(train_generator.class_indices.keys())
n_classes = len(class_names)
print(f'Class names: {class_names}')
print('Num of classes:', n_classes)

# Store the number of images in each set
train_set_size = train_generator.n
validation_set_size = validation_generator.n
test_set_size = test_generator.n
test_full_set_size = test_full_generator.n

print("Train set size:", train_set_size)
print("Validation set size:", validation_set_size)
print("Test set size:", test_set_size)
print('test_full_size:', test_full_set_size)

Found 2017 images belonging to 2 classes.
Found 431 images belonging to 2 classes.
Found 200 images belonging to 2 classes.
Found 435 images belonging to 2 classes.
Class names: ['Colias_alfacariensis', 'Colias_hyale']
Num of classes: 2
Train set size: 2017
Validation set size: 431
Test set size: 200
test_full_size: 435


In [17]:
model = tf.keras.applications.VGG16(input_shape=(224, 224, 3),
                                   weights = 'imagenet',
                                   include_top = False,
                                   )
                                   
X= model.layers[-1].output

# Additonal layers can be added and changed from here
X = tf.keras.layers.GlobalAveragePooling2D()(X)
X = tf.keras.layers.BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001)(X)
X = tf.keras.layers.Dropout(0.25)(X)
X = tf.keras.layers.Dense(288, activation='relu')(X)
X = tf.keras.layers.BatchNormalization()(X)
X = tf.keras.layers.Dropout(0.2)(X)
predictions = Dense(n_classes, activation="softmax")(X)

model_final = Model(model.input, predictions)

In [18]:
for layers in (model.layers)[:-1]:
    print(layers)
    layers.trainable = False
    
for layer in model_final.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

In [19]:
for index, layer in enumerate(model_final.layers):
    print("Layer: {}, Trainable: {}".format(index, layer.trainable))


Layer: 0, Trainable: False
Layer: 1, Trainable: False
Layer: 2, Trainable: False
Layer: 3, Trainable: False
Layer: 4, Trainable: False
Layer: 5, Trainable: False
Layer: 6, Trainable: False
Layer: 7, Trainable: False
Layer: 8, Trainable: False
Layer: 9, Trainable: False
Layer: 10, Trainable: False
Layer: 11, Trainable: False
Layer: 12, Trainable: False
Layer: 13, Trainable: False
Layer: 14, Trainable: False
Layer: 15, Trainable: False
Layer: 16, Trainable: False
Layer: 17, Trainable: False
Layer: 18, Trainable: True
Layer: 19, Trainable: True
Layer: 20, Trainable: False
Layer: 21, Trainable: True
Layer: 22, Trainable: True
Layer: 23, Trainable: False
Layer: 24, Trainable: True
Layer: 25, Trainable: True


In [20]:
model_final.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [21]:
checkpoint_path = f'./dataset_C_seg_results/best_model_{date}.keras'
checkpoint_dir = os.path.dirname(checkpoint_path)

cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_path,save_best_only = True, monitor='val_accuracy' ,save_weights_only = False, verbose = 1)
early = EarlyStopping(monitor='val_accuracy', min_delta=0.01, patience=2, verbose=4, mode='max')

base_learning_rate = 0.0005

In [22]:
model_final.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
              loss=tf.keras.losses.categorical_crossentropy,
              metrics=['accuracy',
                                 ])

In [23]:
initial_epochs = 3

In [24]:
def lr_exp_decay(initial_epochs, lr):
    k = 0.01
    return base_learning_rate * math.exp(-k*initial_epochs)

In [25]:
history = model_final.fit(train_generator,
                          epochs= initial_epochs,
                          validation_data= validation_generator,
                          callbacks=[cp_callback,
                                     early
                                     ]
                         )


Epoch 1/3
127/127 [==============================] - ETA: 0s - loss: 0.5290 - accuracy: 0.7630
Epoch 1: val_accuracy improved from -inf to 0.79814, saving model to ./dataset_C_seg_results\best_model_2026_08_07-05:29_28_AM.keras
127/127 [==============================] - 91s 704ms/step - loss: 0.5290 - accuracy: 0.7630 - val_loss: 0.3681 - val_accuracy: 0.7981
Epoch 2/3
127/127 [==============================] - ETA: 0s - loss: 0.3709 - accuracy: 0.8344
Epoch 2: val_accuracy improved from 0.79814 to 0.87239, saving model to ./dataset_C_seg_results\best_model_2026_08_07-05:29_28_AM.keras
127/127 [==============================] - 80s 635ms/step - loss: 0.3709 - accuracy: 0.8344 - val_loss: 0.2586 - val_accuracy: 0.8724
Epoch 3/3
127/127 [==============================] - ETA: 0s - loss: 0.3131 - accuracy: 0.8686
Epoch 3: val_accuracy improved from 0.87239 to 0.93736, saving model to ./dataset_C_seg_results\best_model_2026_08_07-05:29_28_AM.keras
127/127 [==============================] -

In [26]:
test_loss, test_acc = model_final.evaluate(
    test_generator)

13/13 [==============================] - 7s 571ms/step - loss: 0.4499 - accuracy: 0.7950


In [27]:
for layer in model_final.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

for index, layer in enumerate(model_final.layers):
    print("Layer: {}, Trainable: {}".format(index, layer.trainable))

Layer: 0, Trainable: True
Layer: 1, Trainable: True
Layer: 2, Trainable: True
Layer: 3, Trainable: True
Layer: 4, Trainable: True
Layer: 5, Trainable: True
Layer: 6, Trainable: True
Layer: 7, Trainable: True
Layer: 8, Trainable: True
Layer: 9, Trainable: True
Layer: 10, Trainable: True
Layer: 11, Trainable: True
Layer: 12, Trainable: True
Layer: 13, Trainable: True
Layer: 14, Trainable: True
Layer: 15, Trainable: True
Layer: 16, Trainable: True
Layer: 17, Trainable: True
Layer: 18, Trainable: True
Layer: 19, Trainable: True
Layer: 20, Trainable: False
Layer: 21, Trainable: True
Layer: 22, Trainable: True
Layer: 23, Trainable: False
Layer: 24, Trainable: True
Layer: 25, Trainable: True


In [28]:
# Let's take a look to see how many layers are in the base model
print("Number of layers in the base model: ", len(model_final.layers))

# Fine-tune from this layer onwards
fine_tune_at = 8

# Freeze all the layers before the `fine_tune_at` layer
for layer in model_final.layers[:fine_tune_at]:
  layer.trainable = False

for layer in model_final.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False


Number of layers in the base model:  26


In [29]:
model_final.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [30]:
for index, layer in enumerate(model_final.layers):
    print("Layer: {}, Trainable: {}".format(index, layer.trainable))

Layer: 0, Trainable: False
Layer: 1, Trainable: False
Layer: 2, Trainable: False
Layer: 3, Trainable: False
Layer: 4, Trainable: False
Layer: 5, Trainable: False
Layer: 6, Trainable: False
Layer: 7, Trainable: False
Layer: 8, Trainable: True
Layer: 9, Trainable: True
Layer: 10, Trainable: True
Layer: 11, Trainable: True
Layer: 12, Trainable: True
Layer: 13, Trainable: True
Layer: 14, Trainable: True
Layer: 15, Trainable: True
Layer: 16, Trainable: True
Layer: 17, Trainable: True
Layer: 18, Trainable: True
Layer: 19, Trainable: True
Layer: 20, Trainable: False
Layer: 21, Trainable: True
Layer: 22, Trainable: True
Layer: 23, Trainable: False
Layer: 24, Trainable: True
Layer: 25, Trainable: True


In [31]:
# learning rate has been lowered as more model ahs been opened for training. Should stop overfititng

TL_learningRate = base_learning_rate/10

model_final.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=TL_learningRate),
              loss=tf.keras.losses.categorical_crossentropy,
              metrics=['accuracy'])

In [32]:
fine_tune_epochs = 200

total_epochs = initial_epochs + fine_tune_epochs

#np.random.seed(343)
# fit the model
history_fine = model_final.fit(
  train_generator,
  epochs=total_epochs,
  initial_epoch = history.epoch[-1],
  #initial_epoch = fine_tune_epochs,
  validation_data=validation_generator,
  callbacks=[cp_callback, early,
             ])


Epoch 3/203
127/127 [==============================] - ETA: 0s - loss: 0.4167 - accuracy: 0.8245
Epoch 3: val_accuracy improved from 0.93736 to 0.97448, saving model to ./dataset_C_seg_results\best_model_2026_08_07-05:29_28_AM.keras
127/127 [==============================] - 54s 415ms/step - loss: 0.4167 - accuracy: 0.8245 - val_loss: 0.0919 - val_accuracy: 0.9745
Epoch 4/203
127/127 [==============================] - ETA: 0s - loss: 0.1727 - accuracy: 0.9400
Epoch 4: val_accuracy did not improve from 0.97448
127/127 [==============================] - 53s 417ms/step - loss: 0.1727 - accuracy: 0.9400 - val_loss: 0.1457 - val_accuracy: 0.9490
Epoch 5/203
127/127 [==============================] - ETA: 0s - loss: 0.1385 - accuracy: 0.9539
Epoch 5: val_accuracy did not improve from 0.97448
127/127 [==============================] - 58s 458ms/step - loss: 0.1385 - accuracy: 0.9539 - val_loss: 0.0998 - val_accuracy: 0.9629
Epoch 5: early stopping


In [33]:
test_loss, test_acc = model_final.evaluate(
    test_generator)

13/13 [==============================] - 4s 317ms/step - loss: 0.3852 - accuracy: 0.8150


In [34]:
acc_str = str(round(test_acc * 100,2))


model_name = f'./dataset_C_seg_results/final_model_{acc_str}_{date}.h5'

model_final.save(model_name)

In [35]:
# Make predictions on the test set
predictions = model_final.predict(test_generator)

# Convert predictions to class labels
predicted_class_indices = np.argmax(predictions, axis=1)
predicted_classes = [class_names[idx] for idx in predicted_class_indices]

# Get true class labels
true_classes = test_generator.classes
class_labels = list(test_generator.class_indices.keys())

# Compute confusion matrix and classification report
confusion_mtx = confusion_matrix(true_classes, predicted_class_indices)
class_report = classification_report(true_classes, predicted_class_indices, target_names=class_labels, output_dict=True)

# Convert confusion matrix and classification report to DataFrame
confusion_mtx_df = pd.DataFrame(confusion_mtx, index=class_labels, columns=class_labels)
class_report_df = pd.DataFrame(class_report).transpose()

# Save confusion matrix and classification report to Excel
with pd.ExcelWriter('./dataset_C_seg_results/classification_results_test_even.xlsx') as writer:  
    confusion_mtx_df.to_excel(writer, sheet_name='Confusion Matrix')
    class_report_df.to_excel(writer, sheet_name='Classification Report')


13/13 [==============================] - 4s 303ms/step


In [36]:
# Make predictions on the test set
predictions = model_final.predict(test_full_generator)

# Convert predictions to class labels
predicted_class_indices = np.argmax(predictions, axis=1)
predicted_classes = [class_names[idx] for idx in predicted_class_indices]

# Get true class labels
true_classes = test_full_generator.classes
class_labels = list(test_full_generator.class_indices.keys())

# Compute confusion matrix and classification report
confusion_mtx = confusion_matrix(true_classes, predicted_class_indices)
class_report = classification_report(true_classes, predicted_class_indices, target_names=class_labels, output_dict=True)

# Convert confusion matrix and classification report to DataFrame
confusion_mtx_df = pd.DataFrame(confusion_mtx, index=class_labels, columns=class_labels)
class_report_df = pd.DataFrame(class_report).transpose()

# Save confusion matrix and classification report to Excel
with pd.ExcelWriter('./dataset_C_seg_results/classification_results_test_full.xlsx') as writer:  
    confusion_mtx_df.to_excel(writer, sheet_name='Confusion Matrix')
    class_report_df.to_excel(writer, sheet_name='Classification Report')

28/28 [==============================] - 14s 496ms/step


In [37]:
from PIL import Image

# Set the path to the folder containing the images
img_folder_path = './dataset_C_seg_split/test_even/'

# Set the path to the folder where the predicted images will be saved
output_folder_path = './dataset_C_seg_results/predicted_images/'

# Create an ImageDataGenerator instance
datagen = ImageDataGenerator()
generator = datagen.flow_from_directory(img_folder_path, target_size=(img_width, img_height), batch_size=1, shuffle=False, class_mode='categorical')

# Initialize an empty list to store the results
results_list = []

# Iterate through the subdirectories in the folder
for sub_dir_name in os.listdir(img_folder_path):
    sub_dir_path = os.path.join(img_folder_path, sub_dir_name)
    if os.path.isdir(sub_dir_path):
        # Create a corresponding subdirectory in the output folder
        output_sub_dir_path = os.path.join(output_folder_path, sub_dir_name)
        if not os.path.exists(output_sub_dir_path):
            os.makedirs(output_sub_dir_path)

        # Iterate through the images in the subdirectory
        for img_file in os.listdir(sub_dir_path):
            if img_file.endswith(('.jpg', '.jpeg', '.png')):  # Consider only JPG files
                # Read the image, convert to RGB color space, and resize
                img_path = os.path.join(sub_dir_path, img_file)
                with Image.open(img_path) as img:
                    img = img.convert('RGB')
                    img = img.resize((img_width, img_height))

                # Make a prediction on the image
                img_array = np.asarray(img)
                img_array = preprocess_image(img_array)
                prediction = model_final.predict(np.expand_dims(img_array, axis=0))[0]
                predicted_class_idx = np.argmax(prediction)
                predicted_class = class_names[predicted_class_idx]
                confidence_score = prediction[predicted_class_idx]

                # Save a copy of the image to the corresponding predicted class subdirectory
                output_sub_dir_class_path = os.path.join(output_sub_dir_path, str(predicted_class_idx))
                if not os.path.exists(output_sub_dir_class_path):
                    os.makedirs(output_sub_dir_class_path)
                output_img_path = os.path.join(output_sub_dir_class_path, img_file)
                img.save(output_img_path)

                # Add the result to the list
                actual_class = generator.class_indices[sub_dir_name]
                result_dict = {
                    'Image': img_file,
                    'Actual_Class': actual_class,
                    'Predicted_Class': predicted_class,
                    'Confidence_Score': confidence_score
                }
                results_list.append(result_dict)

# Create a DataFrame from the results list
results_df = pd.DataFrame(results_list)

# Save the results to a CSV file named 'predicted_results.csv'
# results_df.to_csv('../save_offs/FV_dorsal/predicted_images/predicted_results.csv', index=False)
results_df.to_csv(os.path.join(output_folder_path, 'predicted_results.csv'), index=False)


Found 200 images belonging to 2 classes.
1/1 [==============================] - 0s 19ms/step


In [38]:
import tensorflow.keras.backend as K
replace2linear = ReplaceToLinear()

gradcam = Gradcam(model_final,
                  model_modifier=replace2linear,
                  clone=True)

img_folder_path = './dataset_C_seg_results/predicted_images/'

# Create heatmap directory
heatmap_dir = './dataset_C_seg_results/true_heatmaps/'
os.makedirs(heatmap_dir, exist_ok=True)

# Initialize an empty DataFrame
df = pd.DataFrame(columns=['Image_Name', 'Subdirectory', 'Class_Index', 'Heatmap_Intensity'])

BATCHSIZE = 32

for sub_dir_name in os.listdir(img_folder_path):
    sub_dir_path = os.path.join(img_folder_path, sub_dir_name)
    if os.path.isdir(sub_dir_path):
        for sub_sub_dir_name in os.listdir(sub_dir_path):
            sub_sub_dir_path = os.path.join(sub_dir_path, sub_sub_dir_name)
            if os.path.isdir(sub_sub_dir_path):
                class_index = int(sub_sub_dir_name)
                print(f"Processing images in sub-subdirectory {sub_sub_dir_path}, using class index {class_index}")

                def score_function(output):
                    return output[0][class_index]

                img_files = [img_file for img_file in os.listdir(sub_sub_dir_path) 
                             if img_file.lower().endswith(('.jpg', '.jpeg', '.png'))]
                print(f"Found {len(img_files)} images in {sub_sub_dir_path}")

                if len(img_files) == 0:
                    print("No images found, skipping.")
                    continue

                for i in range(0, len(img_files), BATCHSIZE):
                    batch_img_files = img_files[i:i+BATCHSIZE]
                    for img_file in batch_img_files:
                        img_path = os.path.join(sub_sub_dir_path, img_file)
                        try:
                            img = load_img(img_path, target_size=(224, 224))
                            x = np.array(img)
                            x = np.expand_dims(x, axis=0)
                            x = preprocess_input(x)

                            cam = gradcam(score_function, x, penultimate_layer=-1)

                            heatmap_quantified = np.sum(cam[0])

                            df = pd.concat([df, pd.DataFrame([{'Image_Name': img_file, 'Subdirectory': sub_dir_name, 'Class_Index': class_index, 'Heatmap_Intensity': heatmap_quantified}])], ignore_index=True)

                            heatmap = np.uint8(cm.jet(cam[0])[..., :3] * 255)
                            plt.imshow(img)
                            plt.imshow(heatmap, cmap='jet', alpha=0.5)
                            plt.axis('off')
                            plt.title(f"{img_file}, Heatmap intensity: {heatmap_quantified}")

                            heatmap_save_dir = os.path.join(heatmap_dir, sub_dir_name, sub_sub_dir_name)
                            os.makedirs(heatmap_save_dir, exist_ok=True)
                            save_path = os.path.join(heatmap_save_dir, f"{img_file}_heatmap.jpg")
                            plt.savefig(save_path)
                            plt.close()
                        except Exception as e:
                            print(f"Error processing {img_file}: {e}")

                    K.clear_session()

df = df[['Image_Name', 'Subdirectory', 'Class_Index', 'Heatmap_Intensity']]
excel_file_path = "./dataset_C_seg_results/heatmap_results.xlsx"
df.to_excel(excel_file_path, index=False)

Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\0, using class index 0
Found 88 images in ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\0


C:\Users\lenovo\AppData\Local\Temp\ipykernel_59260\2771834629.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{'Image_Name': img_file, 'Subdirectory': sub_dir_name, 'Class_Index': class_index, 'Heatmap_Intensity': heatmap_quantified}])], ignore_index=True)


Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\1, using class index 1
Found 12 images in ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\1
Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_hyale\0, using class index 0
Found 26 images in ./dataset_C_seg_results/predicted_images/Colias_hyale\0
Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_hyale\1, using class index 1
Found 74 images in ./dataset_C_seg_results/predicted_images/Colias_hyale\1


In [39]:
replace2linear = ReplaceToLinear()

# Create the Saliency object
saliency = Saliency(model_final,
                    model_modifier=replace2linear,
                    clone=True)

img_folder_path = './dataset_C_seg_results/predicted_images/'

# Create heatmap directory
heatmap_dir = './dataset_C_seg_results/true_saliency_maps/'
os.makedirs(heatmap_dir, exist_ok=True)

# Initialize an empty DataFrame
df = pd.DataFrame(columns=['Image_Name', 'Subdirectory', 'Class_Index', 'Saliency_Intensity'])

BATCHSIZE = 32  # You can adjust this value based on your available memory

for sub_dir_name in os.listdir(img_folder_path):
    sub_dir_path = os.path.join(img_folder_path, sub_dir_name)
    if os.path.isdir(sub_dir_path):
        for sub_sub_dir_name in os.listdir(sub_dir_path):
            sub_sub_dir_path = os.path.join(sub_dir_path, sub_sub_dir_name)
            if os.path.isdir(sub_sub_dir_path):
                class_index = int(sub_sub_dir_name)
                print(f"Processing images in sub-subdirectory {sub_sub_dir_path}, using class index {class_index}")

                # define score_function in the loop to capture current class_index
                def score_function(output):
                    return output[0][class_index]

                # obtain all images in current directory (.jpg, .jpeg, .png)
                img_files = [img_file for img_file in os.listdir(sub_sub_dir_path) 
                             if img_file.lower().endswith(('.jpg', '.jpeg', '.png'))]
                print(f"Found {len(img_files)} images in {sub_sub_dir_path}")

                if len(img_files) == 0:
                    print("No images found, skipping.")
                    continue

                # Process images in batches
                for i in range(0, len(img_files), BATCHSIZE):
                    batch_img_files = img_files[i:i+BATCHSIZE]
                    for img_file in batch_img_files:
                        img_path = os.path.join(sub_sub_dir_path, img_file)
                        try:
                            # Load the image at original size
                            img = Image.open(img_path)
                            original_size = img.size

                            # Resize image to model input size for saliency
                            img_resized = img.resize((224, 224))
                            x = img_to_array(img_resized)
                            x = np.expand_dims(x, axis=0)
                            x = preprocess_input(x)

                            # Generate the saliency map
                            saliency_map = saliency(score_function,
                                                    x,
                                                    smooth_samples=20,  # Number of gradient iterations
                                                    smooth_noise=0.20)  # Noise spread level

                            # Calculate saliency map intensity as a sum
                            saliency_intensity = np.sum(saliency_map[0])

                            # Add results to dataframe
                            df = pd.concat([df, pd.DataFrame([{'Image_Name': img_file, 'Subdirectory': sub_dir_name, 'Class_Index': class_index, 'Saliency_Intensity': saliency_intensity}])], ignore_index=True)

                            # Resize the saliency map to the original image size
                            saliency_resized = np.uint8(cm.jet(saliency_map[0])[..., :3] * 255)
                            saliency_resized = Image.fromarray(saliency_resized).resize(original_size)

                            # Overlay saliency map on original image
                            plt.imshow(img)
                            plt.imshow(saliency_resized, cmap='jet', alpha=0.5)
                            plt.axis('off')
                            plt.title(f"{img_file}, Saliency intensity: {saliency_intensity}")

                            # Save the image
                            saliency_save_dir = os.path.join(heatmap_dir, sub_dir_name, sub_sub_dir_name)
                            os.makedirs(saliency_save_dir, exist_ok=True)
                            save_path = os.path.join(saliency_save_dir, f"{img_file}_saliency.jpg")
                            plt.savefig(save_path)
                            plt.close()
                        except Exception as e:
                            print(f"Error processing {img_file}: {e}")

                    # After processing each batch, clear memory
                    K.clear_session()

# Reorder the DataFrame columns
df = df[['Image_Name', 'Subdirectory', 'Class_Index', 'Saliency_Intensity']]

# Define the path where you want to save the Excel file
excel_file_path = "./dataset_C_seg_results/saliency_results.xlsx"

# Save DataFrame to Excel
df.to_excel(excel_file_path, index=False)

Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\0, using class index 0
Found 88 images in ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\0


C:\Users\lenovo\AppData\Local\Temp\ipykernel_59260\3242542686.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{'Image_Name': img_file, 'Subdirectory': sub_dir_name, 'Class_Index': class_index, 'Saliency_Intensity': saliency_intensity}])], ignore_index=True)


Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\1, using class index 1
Found 12 images in ./dataset_C_seg_results/predicted_images/Colias_alfacariensis\1
Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_hyale\0, using class index 0
Found 26 images in ./dataset_C_seg_results/predicted_images/Colias_hyale\0
Processing images in sub-subdirectory ./dataset_C_seg_results/predicted_images/Colias_hyale\1, using class index 1
Found 74 images in ./dataset_C_seg_results/predicted_images/Colias_hyale\1


In [40]:
# Define tuning result storage directory 
from pathlib import Path
TUNING_ROOT = Path("./dataset_C_seg_results")
TUNER_DIRECTORY = TUNING_ROOT / "keras_tuner"
TENSORBOARD_DIRECTORY = TUNING_ROOT / "tensorboard" / datetime.now().strftime("%Y%m%d_%H%M%S")

# Ensure directories exist
TUNER_DIRECTORY.mkdir(parents=True, exist_ok=True)
TENSORBOARD_DIRECTORY.parent.mkdir(parents=True, exist_ok=True)

print(f"Tuner log directory: {TUNER_DIRECTORY}")
print(f"TensorBoard log directory: {TENSORBOARD_DIRECTORY}")

Tuner log directory: dataset_C_seg_results\keras_tuner
TensorBoard log directory: dataset_C_seg_results\tensorboard\20260807_054012


In [41]:
# Model building function (adapted for current binary classification, using img_width, img_height, n_classes already defined)
def build_model(hp):
    # Load pretrained VGG16 without top
    base_model = VGG16(input_shape=(img_width, img_height, 3), weights='imagenet', include_top=False)
    # Freeze all convolutional layers
    for layer in base_model.layers:
        layer.trainable = False

    inputs = keras.Input(shape=(img_width, img_height, 3))
    x = base_model(inputs, training=False)  # training=False to properly freeze BatchNormalization behavior
    
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001)(x)
    x = Dropout(hp.Float('dropout', 0, 0.9, step=0.05, default=0.2))(x)
    
    x = Dense(units=hp.Int('units', 32, 512, step=32, default=256),
              activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(hp.Float('dense_dropout', 0, 0.9, step=0.05, default=0.65))(x)
    
    outputs = Dense(n_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice('learning_rate', values=[1e-2, 5e-2, 1e-3, 5e-3, 1e-4, 5e-4, 1e-6, 5e-6], default=1e-3)
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [42]:
# Initialize Hyperband tuner (max_epochs=10 for quick trial, change back to 20 for full tuning)
tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=20,   # Set to 20 for full tuning
    factor=3,
    directory=str(TUNER_DIRECTORY),
    project_name=f"colias_tuning_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    overwrite=True,  # Overwrite previous tuning records on each run
)

# Callbacks (TensorBoard monitoring)
callbacks = [
    TensorBoard(log_dir=str(TENSORBOARD_DIRECTORY), update_freq="epoch"),
]

In [43]:
# Start search
print("Starting hyperparameter search (may take some time, please be patient)...")
tuner.search(
    train_generator,
    validation_data=validation_generator,
    epochs=10,       # Must match max_epochs
    verbose=1,
    callbacks=callbacks,
)

# Check for completed trials
completed_trials = [
    trial for trial in tuner.oracle.trials.values()
    if trial.status == "COMPLETED" and trial.score is not None
]
if not completed_trials:
    raise RuntimeError("Tuning did not complete or no valid trial found. Please check data loading.")
print(f" Completed trials: {len(completed_trials)}")

# Print best hyperparameters
best_hps = tuner.get_best_hyperparameters(1)[0]
print("\n Best hyperparameters:")
for name, value in best_hps.values.items():
    if not name.startswith("tuner/"):
        print(f"  {name}: {value}")

# Get best model
best_model = tuner.get_best_models(1)[0]

# Evaluate best model on test set
print("\n Evaluating best model on test set (test_even)...")
test_metrics = best_model.evaluate(test_generator, return_dict=True)
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test loss: {test_metrics['loss']:.4f}")

# Save best model in .keras format
best_model_save_path = f"./dataset_C_seg_results/best_tuned_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.keras"
best_model.save(best_model_save_path)
print(f" Best tuned model saved to: {best_model_save_path}")

Trial 30 Complete [00h 13m 19s]
val_accuracy: 0.7215777635574341

Best val_accuracy So Far: 0.9814385771751404
Total elapsed time: 03h 05m 32s
 Completed trials: 30

 Best hyperparameters:
  dropout: 0.8
  units: 416
  dense_dropout: 0.8
  learning_rate: 0.0001

 Evaluating best model on test set (test_even)...
13/13 [==============================] - 4s 255ms/step - loss: 0.3785 - accuracy: 0.8150
Test accuracy: 0.8150
Test loss: 0.3785
 Best tuned model saved to: ./dataset_C_seg_results/best_tuned_model_20260807_084550.keras


In [44]:
# ### Generate Grad-CAM and Saliency maps using the trained model (model_final)

import tensorflow.keras.backend as K
replace2linear = ReplaceToLinear()

# ----- Grad-CAM -----
print("\n--- Generating Grad-CAM heatmaps using model_final ---")
gradcam = Gradcam(model_final, model_modifier=replace2linear, clone=True)

images_dir = './dataset_C_seg_results/predicted_images/'
heatmap_dir = './dataset_C_seg_results/true_heatmaps_tuned/'
os.makedirs(heatmap_dir, exist_ok=True)

df_heat = pd.DataFrame(columns=['Image_Name', 'Subdirectory', 'Class_Index', 'Heatmap_Intensity'])
BATCHSIZE = 32

for sub_dir_name in os.listdir(images_dir):
    sub_dir_path = os.path.join(images_dir, sub_dir_name)
    if not os.path.isdir(sub_dir_path):
        continue
    for sub_sub_dir_name in os.listdir(sub_dir_path):
        sub_sub_dir_path = os.path.join(sub_dir_path, sub_sub_dir_name)
        if not os.path.isdir(sub_sub_dir_path):
            continue
        class_index = int(sub_sub_dir_name)
        print(f"Processing {sub_dir_name}/{sub_sub_dir_name} (class {class_index})")
        
        def score_function(output):
            return output[0][class_index]
        
        img_files = [f for f in os.listdir(sub_sub_dir_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if not img_files:
            continue
        
        for i in range(0, len(img_files), BATCHSIZE):
            batch = img_files[i:i+BATCHSIZE]
            for img_file in batch:
                img_path = os.path.join(sub_sub_dir_path, img_file)
                try:
                    img = load_img(img_path, target_size=(224,224))
                    x = np.array(img, dtype=np.float32)
                    x = np.expand_dims(x, axis=0)
                    x = preprocess_input(x)
                    
                    cam = gradcam(score_function, x, penultimate_layer=-1)
                    heat_val = np.sum(cam[0])
                    
                    df_heat = pd.concat([df_heat, pd.DataFrame([{
                        'Image_Name': img_file,
                        'Subdirectory': sub_dir_name,
                        'Class_Index': class_index,
                        'Heatmap_Intensity': heat_val
                    }])], ignore_index=True)
                    
                    heatmap = np.uint8(cm.jet(cam[0])[..., :3] * 255)
                    plt.imshow(img)
                    plt.imshow(heatmap, cmap='jet', alpha=0.5)
                    plt.axis('off')
                    plt.title(f"{img_file} (intensity: {heat_val:.2f})")
                    
                    save_dir = os.path.join(heatmap_dir, sub_dir_name, sub_sub_dir_name)
                    os.makedirs(save_dir, exist_ok=True)
                    save_path = os.path.join(save_dir, f"{img_file}_heatmap.jpg")
                    plt.savefig(save_path)
                    plt.close()
                except Exception as e:
                    print(f"Error on {img_file}: {e}")
            K.clear_session()

df_heat = df_heat[['Image_Name', 'Subdirectory', 'Class_Index', 'Heatmap_Intensity']]
heat_excel = os.path.join('./dataset_C_seg_results/', 'heatmap_results_tuned.xlsx')
df_heat.to_excel(heat_excel, index=False)
print(f"Grad-CAM heatmaps saved to {heatmap_dir}")

# ----- Saliency -----
print("\n--- Generating Saliency maps using model_final ---")
saliency = Saliency(model_final, model_modifier=replace2linear, clone=True)

saliency_dir = './dataset_C_seg_results/true_saliency_maps_tuned/'
os.makedirs(saliency_dir, exist_ok=True)

df_sal = pd.DataFrame(columns=['Image_Name', 'Subdirectory', 'Class_Index', 'Saliency_Intensity'])

for sub_dir_name in os.listdir(images_dir):
    sub_dir_path = os.path.join(images_dir, sub_dir_name)
    if not os.path.isdir(sub_dir_path):
        continue
    for sub_sub_dir_name in os.listdir(sub_dir_path):
        sub_sub_dir_path = os.path.join(sub_dir_path, sub_sub_dir_name)
        if not os.path.isdir(sub_sub_dir_path):
            continue
        class_index = int(sub_sub_dir_name)
        print(f"Processing {sub_dir_name}/{sub_sub_dir_name} (class {class_index})")
        
        def score_function(output):
            return output[0][class_index]
        
        img_files = [f for f in os.listdir(sub_sub_dir_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if not img_files:
            continue
        
        for i in range(0, len(img_files), BATCHSIZE):
            batch = img_files[i:i+BATCHSIZE]
            for img_file in batch:
                img_path = os.path.join(sub_sub_dir_path, img_file)
                try:
                    from PIL import Image
                    img_orig = Image.open(img_path)
                    original_size = img_orig.size
                    
                    img_resized = img_orig.resize((224,224))
                    x = img_to_array(img_resized)
                    x = np.expand_dims(x, axis=0)
                    x = preprocess_input(x)
                    
                    sal_map = saliency(score_function, x, smooth_samples=20, smooth_noise=0.20)
                    sal_val = np.sum(sal_map[0])
                    
                    df_sal = pd.concat([df_sal, pd.DataFrame([{
                        'Image_Name': img_file,
                        'Subdirectory': sub_dir_name,
                        'Class_Index': class_index,
                        'Saliency_Intensity': sal_val
                    }])], ignore_index=True)
                    
                    sal_resized = np.uint8(cm.jet(sal_map[0])[..., :3] * 255)
                    sal_resized = Image.fromarray(sal_resized).resize(original_size)
                    
                    plt.imshow(img_orig)
                    plt.imshow(sal_resized, cmap='jet', alpha=0.5)
                    plt.axis('off')
                    plt.title(f"{img_file} (intensity: {sal_val:.2f})")
                    
                    save_dir = os.path.join(saliency_dir, sub_dir_name, sub_sub_dir_name)
                    os.makedirs(save_dir, exist_ok=True)
                    save_path = os.path.join(save_dir, f"{img_file}_saliency.jpg")
                    plt.savefig(save_path)
                    plt.close()
                except Exception as e:
                    print(f"Error on {img_file}: {e}")
            K.clear_session()

df_sal = df_sal[['Image_Name', 'Subdirectory', 'Class_Index', 'Saliency_Intensity']]
sal_excel = os.path.join('./dataset_C_seg_results/', 'saliency_results_tuned.xlsx')
df_sal.to_excel(sal_excel, index=False)
print(f"Saliency maps saved to {saliency_dir}")
print("All heatmaps and saliency maps for the tuned model have been generated.")


--- Generating Grad-CAM heatmaps using model_final ---
Processing Colias_alfacariensis/0 (class 0)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_59260\326912350.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_heat = pd.concat([df_heat, pd.DataFrame([{


Processing Colias_alfacariensis/1 (class 1)
Processing Colias_hyale/0 (class 0)
Processing Colias_hyale/1 (class 1)
Grad-CAM heatmaps saved to ./dataset_C_seg_results/true_heatmaps_tuned/

--- Generating Saliency maps using model_final ---
Processing Colias_alfacariensis/0 (class 0)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_59260\326912350.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_sal = pd.concat([df_sal, pd.DataFrame([{


Processing Colias_alfacariensis/1 (class 1)
Processing Colias_hyale/0 (class 0)
Processing Colias_hyale/1 (class 1)
Saliency maps saved to ./dataset_C_seg_results/true_saliency_maps_tuned/
All heatmaps and saliency maps for the tuned model have been generated.
